In [2]:
import json

from openai import OpenAI
from src.models.model_utils import get_api_key
from src.config import PATH_DATA

api_key = get_api_key("openai")
client = OpenAI(api_key=api_key)
input_filename = "openai_gpt-4.1-batch-submit_reddit.jsonl"
input_filepath = PATH_DATA / input_filename

In [2]:
# Verify batch input file
with open(input_filepath, 'r') as f:
    print(f.readline())
    print(f.readline())
with open(input_filepath, 'r') as f:
    num_lines = 0
    for line in f:
        num_lines += 1
    print(num_lines)

{"custom_id": "yfds8s_none", "method": "POST", "url": "/v1/chat/completions", "body": {"model": "gpt-4.1", "messages": [{"role": "system", "content": "You are a tool for evaluating moral dilemmas.\n\nPlease evaluate the following post from the subreddit \"Am I the Asshole\".\n\nFirst, determine whether the OP (original poster) of this post is the asshole in the scenario they describe.\nProvide a categorical label indicating your assessment of the scenario, from one of these choices:\n\n- YTA, which stands for \"You're the Asshole\", is for scenarios where the OP is at fault in their situation.\n- NTA, which stands for \"Not the Asshole\", is for scenarios where the OP is NOT to blame and the other party described in their scenario is to blame.\n- ESH, which stands for \"Everyone Sucks Here\", is for scenarios where both parties are to blame: both people involved in the scenario should be held responsible.\n- NAH, which stands for \"No Assholes Here\", is for scenarios where neither par

In [3]:
# Upload batch input file
batch_input_file = client.files.create(
    file=open(input_filepath, "rb"),
    purpose="batch"
)
print(batch_input_file)
input_file_id = batch_input_file.id

FileObject(id='file-Xk525AFVphqUU8TfHYETFY', bytes=38147400, created_at=1773860877, filename='openai_gpt-4.1-batch-submit_reddit.jsonl', object='file', purpose='batch', status='processed', expires_at=1776452877, status_details=None)


In [4]:
# Create batch
response = client.batches.create(
    input_file_id=input_file_id,
    endpoint="/v1/chat/completions",
    completion_window="24h",
    metadata={
        "description": input_filename
    }
)
print(response)
batch_id = response.id

Batch(id='batch_69baf845202481909eaf6caf854c78a4', completion_window='24h', created_at=1773860933, endpoint='/v1/chat/completions', input_file_id='file-Xk525AFVphqUU8TfHYETFY', object='batch', status='validating', cancelled_at=None, cancelling_at=None, completed_at=None, error_file_id=None, errors=None, expired_at=None, expires_at=1773947333, failed_at=None, finalizing_at=None, in_progress_at=None, metadata={'description': 'openai_gpt-4.1-batch-submit_reddit.jsonl'}, model=None, output_file_id=None, request_counts=BatchRequestCounts(completed=0, failed=0, total=0), usage=BatchUsage(input_tokens=0, input_tokens_details=InputTokensDetails(cached_tokens=0), output_tokens=0, output_tokens_details=OutputTokensDetails(reasoning_tokens=0), total_tokens=0))


In [3]:
# Check batch status
batch_id = "batch_69baf845202481909eaf6caf854c78a4"
batch = client.batches.retrieve(batch_id)
print(batch)

Batch(id='batch_69baf845202481909eaf6caf854c78a4', completion_window='24h', created_at=1773860933, endpoint='/v1/chat/completions', input_file_id='file-Xk525AFVphqUU8TfHYETFY', object='batch', status='completed', cancelled_at=None, cancelling_at=None, completed_at=1773863268, error_file_id=None, errors=None, expired_at=None, expires_at=1773947333, failed_at=None, finalizing_at=1773862280, in_progress_at=1773860997, metadata={'description': 'openai_gpt-4.1-batch-submit_reddit.jsonl'}, model='gpt-4.1-2025-04-14', output_file_id='file-Bq7862pb4qZL76bkj8BNJB', request_counts=BatchRequestCounts(completed=8029, failed=0, total=8029), usage=BatchUsage(input_tokens=7809021, input_tokens_details=InputTokensDetails(cached_tokens=0), output_tokens=990513, output_tokens_details=OutputTokensDetails(reasoning_tokens=0), total_tokens=8799534))


In [ ]:
# Check errors
error_file_id = "file-WMSzBa3MTiT1yF3ABTQ5gH"
file_response = client.files.content(error_file_id)
print(file_response.text)

In [ ]:
# Retrieve results: GO TO ONLINE DASHBOARD TO DOWNLOAD
# output_file_id = "file-VcGq5bc4zcjFBtsbJ2Suie"
# file_response = client.files.content(output_file_id)
# 
# output_filename = f"{input_filename}-output.jsonl"
# with open(output_filename, "wb") as f:
#     f.write(file_response.content)
# 
# output = file_response.text.split("\n")
# print(json.dumps(json.loads(output[0]), indent=4))

In [5]:
# Inspect output
output_filename = "openai_gpt-4.1-batch-response_reddit.jsonl"
output_filepath = PATH_DATA / output_filename
with open(output_filepath, "r") as f:
    print(f.readline())

{"id": "batch_req_69bafd8af87881908b5be67e18f8b759", "custom_id": "yfds8s_none", "response": {"status_code": 200, "request_id": "f858f2f1-271d-490a-8b2c-8cc66a40423e", "body": {"id": "chatcmpl-DKqZuDPmvS4BfTCG7MgChO06xmAEH", "object": "chat.completion", "created": 1773861042, "model": "gpt-4.1-2025-04-14", "choices": [{"index": 0, "message": {"role": "assistant", "content": "Verdict: NTA\n\nReasoning: You are not obligated to go on a trip where you would feel like the odd one out, especially given the context of being excluded from the groomsmen group. You communicated your feelings honestly and gave your friends the option to proceed without you, which is reasonable. It's understandable that you would feel uncomfortable being the only non-groomsman in a group that is now centered around the wedding party. Your friends are free to invite whoever they want, but you are also free to make choices that protect your own comfort and well-being. This isn't about resentment; it's about setting